In [13]:
import os
from llama_index.core import KnowledgeGraphIndex
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.graph_stores.neo4j import Neo4jGraphStore

from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from custom_llm import OurLLM
llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
NEO4J_DATABASE = "temp"

graph_store = Neo4jGraphStore(
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    url=NEO4J_URI,
    database=NEO4J_DATABASE,
)

C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_23732\3379812111.py:10: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))


In [ ]:
ew = graph_store.query(
    """
MATCH (n) detach delete n
"""
)

storage_context = StorageContext.from_defaults(graph_store=graph_store)

storage_context2 = StorageContext.from_defaults(persist_dir="storage")
indexi = load_index_from_storage(storage_context2)
query_engine = indexi.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))
response = query_engine.query("Tell me about Tlow something")
print(response)

In [ ]:
ew = graph_store.query(
    """
MATCH (n) detach delete n
"""
)

storage_context = StorageContext.from_defaults(graph_store=graph_store)

# add keyword mappings and nodes manually
# add triplets (subject, relationship, object)

from llama_index.core.node_parser import SentenceSplitter
node_parser = SentenceSplitter()
documents = [
    "John is a data scientist. He works on machine learning projects.",
    "Google developed TensorFlow. TensorFlow is used for deep learning."
]
from llama_index.core import Document
documents = [Document(text=t) for t in documents]

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.indices.knowledge_graph import KnowledgeGraphIndex

# Step 1: Split documents into nodes
node_parser = SentenceSplitter()
nodes = node_parser.get_nodes_from_documents(documents)

# Step 2: Initialize an empty Knowledge Graph Index
index = KnowledgeGraphIndex([], 
                            storage_context=storage_context, 
                            include_embeddings=True)

# Step 3: Manually add keyword mappings (triplets) and nodes
# For node 0 (John)
node_0_tups = [
    ("John", "is a", "data scientist"),
    ("John", "works on", "machine learning projects"),
]
for tup in node_0_tups:
    index.upsert_triplet_and_node(tup, nodes[0], include_embeddings=True)

# For node 1 (Google and TensorFlow)
node_1_tups = [
    ("Google", "developed", "TensorFlow"),
    ("TensorFlow", "is used for", "deep learning"),
]
for tup in node_1_tups:
    index.upsert_triplet_and_node(tup, nodes[1], include_embeddings=True)

# Step 4: Querying the Knowledge Graph
query_engine = index.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))
response = query_engine.query("Tell me about Tlow something")
print(response)


In [ ]:
# Save the index
index.storage_context.persist()

In [ ]:
def parse_triplets(file_path):
    with open(file_path, 'r') as file:
        lines = [line.strip() for line in file if line.strip()]
    
    result = []
    current_set = []
    inside_block = False
    
    for line in lines:
        if line == "jai_balayya_ew":
            if inside_block:
                result.append(current_set)
                current_set = []
            inside_block = not inside_block
        elif inside_block:
            line = line.strip("()")  # Remove parentheses if present
            triplet = tuple(map(str.strip, line.split(',')))
            current_set.append(triplet)
    
    return result

# Example usage:
file_path = "log.txt"
parsed_data = parse_triplets(file_path)



[('rama', 'husband_of', 'sita'), ('lakshmana', 'husband_of', 'urmila'), ('bharata', 'husband_of', 'mandavi'), ('shatrughna', 'husband_of', 'shrutakeerti'), ('rama', 'is_in', 'canto_unknown'), ('sita', 'is_in', 'canto_unknown'), ('lakshmana', 'is_in', 'canto_unknown'), ('urmila', 'is_in', 'canto_unknown'), ('bharata', 'is_in', 'canto_unknown'), ('mandavi', 'is_in', 'canto_unknown'), ('shatrughna', 'is_in', 'canto_unknown'), ('shrutakeerti', 'is_in', 'canto_unknown'), ('janaka', 'is_in', 'canto_unknown'), ('vasistha', 'is_in', 'canto_unknown'), ('rama', 'wedding_ceremony', 'married_sita'), ('lakshmana', 'wedding_ceremony', 'married_urmila'), ('bharata', 'wedding_ceremony', 'married_mandavi'), ('shatrughna', 'wedding_ceremony', 'married_shrutakeerti'), ('janaka', 'oversees', 'wedding_ceremony'), ('vasistha', 'guides', 'wedding_ceremony')]


In [9]:
from llama_index.core import KnowledgeGraphIndex, SimpleDirectoryReader
from llama_index.core import StorageContext
from llama_index.graph_stores.neo4j import Neo4jGraphStore


from llama_index.llms.openai import OpenAI
from IPython.display import Markdown, display

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
# from fastapi import FastAPI
from llama_index.core import StorageContext, load_index_from_storage
from custom_llm import OurLLM
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import BaseSynthesizer

llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))
file = 'balakanda.json'
import json
with open(file) as f:
    data = json.load(f)

from llama_index.core import Document
documents = []
for i in data[:]:       
    document = Document(
    text = i['Prompt_Answer'],
    metadata={
        "canto_summary": i['Summary'],
        "r_id": str(i['Canto']) + "_" + str(i['Prompt_Number'])
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{value}",
    text_template="Content: {content}, Chapter context: {metadata_str}\n",
    )
    documents.append(document)
print(len(documents))
print(f"Document Metadata: {documents[-1].metadata}")
storage_context = StorageContext.from_defaults(graph_store=graph_store)

233
Document Metadata: {'canto_summary': 'Canto LXXVII\n\nOn the departure of Paragurama, Sri Rama hands over the bow of Lord Visnu\nto Varuna (the god of water) and, sending his army ahead, King Dasaratha\nenters Ayodhya. After some days Prince Yudhajit, Bharata’s maternal\nuncle takes away Bharata and Satrughna to his father’s capital\n\n', 'r_id': '77_3'}


C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_23732\3157456374.py:26: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))


In [10]:
from llama_index.core.node_parser import SentenceSplitter
node_parser = SentenceSplitter()
nodes = node_parser.get_nodes_from_documents(documents[219:])

In [12]:
print(len(parsed_data))

14


In [21]:
ew = graph_store.query(
    """
MATCH (n) detach delete n
"""
)

storage_context = StorageContext.from_defaults(graph_store=graph_store)

index = KnowledgeGraphIndex([], 
                            storage_context=storage_context, 
                            include_embeddings=True)
from tqdm import tqdm
for i in tqdm(range(0, len(nodes))):
    for tup in parsed_data[i]:
        index.upsert_triplet_and_node(tup, nodes[i], include_embeddings=True)

# Step 4: Querying the Knowledge Graph
query_engine = index.as_query_engine(
    include_text=True, 
    response_mode="tree_summarize",
    embedding_mode="hybrid",
    similarity_top_k=5,
    verbose=True)
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=2048, context_window = 256, num_output=256))
response = query_engine.query("who are the ministers of dasaratha")
print(response)

C:\Users\M LAKSHMI NARASIMHA\AppData\Local\Temp\ipykernel_23732\3417714044.py:9: DeprecationWarning: Call to deprecated class KnowledgeGraphIndex. (The KnowledgeGraphIndex class has been deprecated. Please use the new PropertyGraphIndex class instead. If a certain graph store integration is missing in the new class, please open an issue on the GitHub repository or contribute it!) -- Deprecated since version 0.10.53.
  index = KnowledgeGraphIndex([],
  0%|          | 0/14 [00:00<?, ?it/s]

 71%|███████▏  | 10/14 [00:30<00:12,  3.09s/it]


KeyboardInterrupt: 

In [18]:
parsed_data[3] = [
 ('dasaratha', 'is_in', 'canto_lxxiv'),
 ('vasistha', 'is_in', 'canto_lxxiv'),
 ('parasurama', 'is_in', 'canto_lxxiv'),
 ('viswamitra', 'is_in', 'canto_lxxiv'),
 ('janaka', 'is_in', 'canto_lxxiv'),
 ('rama', 'is_in', 'canto_lxxiv'),
 ('sita', 'is_in', 'canto_lxxiv'),
 ('dasaratha', 'returning_ayodhya', 'marriage_procession'),
 ('rama', 'returning_ayodhya', 'marriage_procession'),
 ('sita', 'returning_ayodhya', 'marriage_procession'),
 ('viswamitra', 'returning_hermitage', 'after_marriage'),
 ('parasurama', 'confronts', 'dasaratha'),
 ('vasistha', 'offers_worship', 'parasurama'),
 ('parasurama', 'wields', 'axe'),
 ('parasurama', 'known_as', 'kshatriya_enemy'),
 ('dasaratha', 'father_of', 'rama'),
 ('janaka', 'father_of', 'sita'),
 ('sita', 'wife_of', 'rama'),
 ('vasistha', 'mentor_of', 'dasaratha')
 ]

In [20]:
index.storage_context.persist()